# ZelusBench: Selective Attention — Saturated**Extreme distractors (10:1). Heavy filtering.**Fixed distractor level, randomized background conditions(chain depth, DAG structure, transforms, dimensionality).Backgrounds are deterministic per scenario index — shared across all levels.- Distractor ratio: 10:1- 15 scenarios

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import random
import re
import time

from zelusbench.scenarios.config import (
    ScenarioConfig, DAGStructure, QueryType,
    DistractorLevel, TransformDensity,
)
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.scorer import score_query, score_response




DIST_LEVEL = DistractorLevel.EXTREME
RATIO = 10
SEEDS = 15

print(f"ZelusBench Selective Attention — Saturated")
print(f"Ratio: {RATIO}:1 | Seeds: {SEEDS}")

In [ ]:
@kbench.task(name="zelusbench_attn_selective_saturated")
def zelusbench_attn_selective_saturated(llm) -> tuple[float, float]:
    _start = time.time()
    print(f"Running {SEEDS} scenarios (ratio={RATIO}:1)...")
    print("=" * 60)

    all_scores = []

    for i in range(SEEDS):
        bg_rng = random.Random(i * 7919)
        cfg = ScenarioConfig.randomize_except(bg_rng, pinned={
            "distractor_level": DIST_LEVEL,
            "num_queries": 3,
            "seed": RATIO * 1000 + i,
        })
        gen = ScenarioGenerator(cfg)
        scenario = gen.generate(f"selective_{RATIO}_{i}")

        response = llm.prompt(scenario.prompt)
        scores = score_response(response, scenario)
        all_scores.extend(scores)

        avg = float(np.mean([s.score for s in scores]))
        q_depths = [s.chain_depth for s in scores]
        tiers = [s.tier.name for s in scores]
        bg = f"dim={cfg.dim} dag={cfg.dag_structure.name} depth={cfg.min_chain_depth}-{cfg.max_chain_depth} tx={cfg.transform_density.name}"
        print(f"  [{i+1}/{SEEDS}] avg={avg:.2f} q_depths={q_depths} tiers={tiers} | {bg}")

    overall = float(np.mean([s.score for s in all_scores]))
    std = float(np.std([s.score for s in all_scores]))
    elapsed = time.time() - _start
    print(f"\nOverall: {overall:.3f} +/- {std:.3f} | {len(all_scores)} queries | {elapsed:.1f}s")
    kbench.assertions.assert_true(overall >= 0, expectation=f"Ratio {RATIO}:1 valid (got {overall:.3f})")
    return overall, std


zelusbench_attn_selective_saturated.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_attn_selective_saturated